# Library Examples

## Archiving information

In [ ]:
from datetime import datetime, timedelta

import polars as pl
from pytz import UTC

from epicsarchiver import ArchiverAppliance

%matplotlib inline

In [ ]:
import math
import re
import sys

import epicsarchiver.retrieval.EPICSEvent_pb2 as _ee
from epicsarchiver.retrieval.EPICSEvent_pb2 import SCALAR_DOUBLE, PayloadInfo, ScalarDouble
from epicsarchiver.retrieval.pb import escape_bytes
import responses
from aioresponses import aioresponses as _aioresponses_ctx

_YEAR = 2026
_BASE_SECS = 8_640_000


def _make_doc_events(n=30):
    return [
        ScalarDouble(
            secondsintoyear=_BASE_SECS + i * 10,
            nano=int(abs(math.sin(i)) * 1_000_000_000),
            val=28.0 + 0.5 * math.sin(i * 0.4),
            severity=1,
            status=4,
            fieldvalues=[
                _ee.FieldValue(name="EGU", val="degC"),
                _ee.FieldValue(name="PREC", val="2"),
            ],
        )
        for i in range(n)
    ]


def _make_pb(pvname):
    events = _make_doc_events()
    info = PayloadInfo(type=SCALAR_DOUBLE, pvname=pvname, year=_YEAR)
    info_bytes = escape_bytes(info.SerializeToString())
    events_bytes = b"\n".join(escape_bytes(e.SerializeToString()) for e in events)
    return info_bytes + b"\n" + events_bytes


_host = "archiver.example.org"
_retrieval_url = f"http://{_host}:17668/retrieval/data/getData.raw"
_pb_body = _make_pb("EXAMPLE:TEMPERATURE")

_rsps = responses.RequestsMock(assert_all_requests_are_fired=False)
_rsps.add(responses.GET, _retrieval_url, body=_pb_body, status=200)
_rsps.start()

_async_mock = _aioresponses_ctx()
_async_mock.start()
_async_mock.get(
    re.compile(r"http://archiver\.example\.org:17668/.*"),
    body=_pb_body,
    repeat=True,
)

In [ ]:
archiver = ArchiverAppliance("archiver.example.org")
pv = "EXAMPLE:TEMPERATURE"

## Getting Data

In [ ]:
_, events = archiver.get_events(pv, datetime.now(tz=UTC) - timedelta(seconds=1), datetime.now(tz=UTC))
events

In [ ]:
df = archiver.get_data(pv, datetime.now(tz=UTC) - timedelta(seconds=30), datetime.now(tz=UTC))

In [ ]:
df.head()

## Async Fetch Data

In [ ]:
from epicsarchiver.retrieval.client.async_archiver_retrieval import AsyncArchiverRetrieval
from pytz import timezone

In [ ]:
tz = timezone("Europe/Stockholm")

In [ ]:
async with AsyncArchiverRetrieval(archiver.hostname) as a_archiver:
    print(await a_archiver.get_events(pv, datetime.now(tz=tz) - timedelta(microseconds=100), datetime.now(tz=tz)))

In [ ]:
async with AsyncArchiverRetrieval(archiver.hostname) as a_archiver:
    print(await a_archiver.get_all_events([pv, "EXAMPLE:TEMPERATURE2", "EXAMPLE:TEMPERATURE3"], datetime.now(tz=tz) - timedelta(microseconds=100), datetime.now(tz=tz)))

## Displaying and Calculating Summaries

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df["date"], df["val"])
plt.xlabel("time")
plt.ylabel("val")
plt.tight_layout()
# NBVAL_IGNORE_OUTPUT

In [ ]:
from epicsarchiver.retrieval.client.processor import Processor, ProcessorName

In [ ]:
df_mean = archiver.get_data(
    pv,
    datetime.now(tz=UTC) - timedelta(seconds=6000),
    datetime.now(tz=UTC),
    Processor(ProcessorName.MEAN, 20),
)
plt.plot(df_mean["date"], df_mean["val"])
plt.xlabel("time")
plt.ylabel("val (mean, 20s bins)")
plt.tight_layout()
# NBVAL_IGNORE_OUTPUT

In [ ]:
_rsps.stop()
_rsps.reset()
_async_mock.stop()